In [ ]:
# Core imports
import sys
import json
import os
from pathlib import Path

import pandas as pd

_here = Path.cwd()
if (_here / "utils").exists():
    pass
elif (_here / "notebooks" / "utils").exists():
    sys.path.insert(0, str(_here / "notebooks"))

import dspy
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field
from tqdm import tqdm


In [3]:
# DSPy Configuration
lm = dspy.LM(
    model="openai/gpt-5.2", 
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)



### Core Policy Model

In [4]:
from utils.schemas import ExtractedPolicy

In [5]:
from utils.schemas import DocumentMetadata 


---

## 🧠 DSPy Signatures & Modules {#dspy}

Define DSPy signatures and custom modules for policy extraction, validation, and processing.

### Policy Extraction Signature

In [6]:
from utils.dspy_extraction import PolicyExtractionSignature, PolicyExtractor

In [7]:
# (Legacy strict validator removed)
# Individual validation uses the refined validator in `utils.dspy_validation`.
from utils.dspy_validation import PolicyValidationSignature, PolicyValidator, ValidationMetrics

In [8]:
from utils.docling_io import pdf_to_markdown

---

## 🗂️ Batch Configuration

Define the list of cities to process. Add a `DocumentMetadata` + PDF path per city. All 7 steps below will loop over this list automatically.

In [ ]:
# ─── Batch Configuration ────────────────────────────────────────────────────
# Add one entry per city. Everything else runs automatically.
# Outputs are written to OUTPUT_DIR/<CityName_Country>/

from utils.schemas import DocumentMetadata

BATCH = [
    {
        "metadata": DocumentMetadata(country="USA", state_or_province="Illinois", city="Chicago"),
        "pdf_path": "../pdfs/Chicago-CAP-071822.pdf",
    },
    # Uncomment and fill in to add more cities:
    # {
    #     "metadata": DocumentMetadata(country="USA", state_or_province="Texas", city="Austin"),
    #     "pdf_path": "../pdfs/Austin-CAP.pdf",
    # },
    # {
    #     "metadata": DocumentMetadata(country="Senegal", city="Dakar"),
    #     "pdf_path": "../pdfs/Dakar-CAP.pdf",
    # },
]

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def city_key(metadata: DocumentMetadata) -> str:
    """Slug used as filename prefix, e.g. 'Chicago_USA'."""
    return f"{metadata.city}_{metadata.country}".replace(" ", "_")

print(f"Batch configured: {[city_key(e['metadata']) for e in BATCH]}")


2026-02-22 14:42:57,644 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-02-22 14:42:58,172 - INFO - Going to convert document batch...
2026-02-22 14:42:58,187 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-02-22 14:42:58,194 - INFO - Loading plugin 'docling_defaults'
2026-02-22 14:42:58,197 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-02-22 14:42:58,216 - INFO - Loading plugin 'docling_defaults'
2026-02-22 14:42:58,220 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-02-22 14:42:59,184 - INFO - Auto OCR model selected ocrmac.
2026-02-22 14:42:59,188 - INFO - Loading plugin 'docling_defaults'
2026-02-22 14:42:59,193 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2026-02-22 14:43:02,020 - INFO - Accelerator device: 'mps'
2026-02-22 14:43:03,604 - INFO - Loading plugin 'docling_defaul

Converted PDF to markdown: ../pdfs/Chicago-CAP-071822.pdf
Markdown characters: 256905


### Step 1 — PDFs → Markdown

Converts each PDF to markdown using Docling. Results are cached to disk so re-runs skip the conversion.

In [ ]:
# Step 1: Convert PDFs → Markdown (cached per city)
from utils.docling_io import pdf_to_markdown

markdowns = {}  # city_key -> markdown string

for entry in BATCH:
    key = city_key(entry["metadata"])
    md_path = OUTPUT_DIR / f"{key}_markdown.md"

    if md_path.exists():
        print(f"[{key}] Loading cached markdown from {md_path}...")
        with open(md_path, "r") as f:
            markdowns[key] = f.read()
    else:
        print(f"[{key}] Converting PDF: {entry['pdf_path']}")
        markdowns[key] = pdf_to_markdown(entry["pdf_path"], save_markdown_path=str(md_path))

    print(f"[{key}] {len(markdowns[key]):,} chars")


### Step 2 — Extract Policies

Runs the DSPy `PolicyExtractor` LLM call for each city. Saves `{City_Country}_extracted_policies.json` to `outputs/`.

In [ ]:
# Step 2: Extract Policies (LLM call per city)
from utils.dspy_extraction import PolicyExtractor

policy_extractor = PolicyExtractor()
all_extracted = {}  # city_key -> List[ExtractedPolicy]

for entry in BATCH:
    key = city_key(entry["metadata"])
    print(f"\n[{key}] Extracting policies...")

    extracted = policy_extractor(
        document_text=markdowns[key],
        document_metadata=entry["metadata"],
    )
    all_extracted[key] = extracted

    out = OUTPUT_DIR / f"{key}_extracted_policies.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump([p.model_dump() for p in extracted], f, ensure_ascii=False, indent=2)

    print(f"[{key}] {len(extracted)} policies extracted → {out}")


### Step 3 — Cluster Policies

Groups extracted policies by section header into `parent_with_subs`, `individual`, and `orphan_sub` clusters. Saves `{City_Country}_policy_clusters.json`.

In [ ]:
# Step 3: Cluster Policies
from utils.clustering import cluster_policies, summarize_clusters

all_clusters = {}  # city_key -> List[dict]

for entry in BATCH:
    key = city_key(entry["metadata"])
    clusters = cluster_policies(all_extracted[key])
    all_clusters[key] = clusters

    print(f"\n[{key}]")
    summarize_clusters(clusters)

    out = OUTPUT_DIR / f"{key}_policy_clusters.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(clusters, f, default=lambda o: o.model_dump(), ensure_ascii=False, indent=2)
    print(f"  → {out}")


### Step 4 — Build Policy Records

Flattens each city's clusters into a structured DataFrame with consistent columns. Saves `{City_Country}_policy_records.csv`.

In [ ]:
# Step 4: Build Policy Records DataFrames
from utils.clustering import clusters_to_records

FRONT_COLS = [
    "cluster_id", "cluster_type", "role", "section_header", "sector",
    "policy_statement", "parent_statement", "verbatim_text", "extraction_rationale",
]

all_policy_records = {}  # city_key -> List[dict]
all_df_policies = {}     # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    records = clusters_to_records(all_clusters[key])
    df = pd.DataFrame(records)
    df = df[FRONT_COLS + [c for c in df.columns if c not in FRONT_COLS]]

    all_policy_records[key] = records
    all_df_policies[key] = df

    out = OUTPUT_DIR / f"{key}_policy_records.csv"
    df.to_csv(out, index=False)
    print(f"[{key}] {len(df)} records → {out}")


### Step 5 — Validate Individual Policies

Runs `PolicyValidator` on each `individual`-role policy. Flattens Pydantic metrics into columns. Saves `{City_Country}_validated_individual.csv`.

In [ ]:
# Step 5: Validate Individual Policies (LLM call per city)
from utils.dspy_validation import PolicyValidator

validator = PolicyValidator()
all_df_final = {}  # city_key -> pd.DataFrame (flattened metrics)

for entry in BATCH:
    key = city_key(entry["metadata"])
    individual = [r for r in all_policy_records[key] if r.get("role") == "individual"]
    print(f"\n[{key}] Validating {len(individual)} individual policies...")

    validated = []
    for record in tqdm(individual, desc=key):
        try:
            pred = validator(policy_data=record)
            validated.append({**record, **pred.toDict()})
        except Exception as e:
            print(f"  ⚠ Error: {e}")
            continue

    df_val = pd.DataFrame(validated)

    # Flatten Pydantic validation_results column into separate columns
    if "validation_results" in df_val.columns:
        vdata = df_val["validation_results"].apply(
            lambda x: x.model_dump() if hasattr(x, "model_dump") else x.dict()
        )
        df_metrics = pd.json_normalize(vdata)
        df_val = pd.concat([df_val.drop(columns=["validation_results"]), df_metrics], axis=1)

    all_df_final[key] = df_val

    out = OUTPUT_DIR / f"{key}_validated_individual.csv"
    df_val.to_csv(out, index=False)
    print(f"[{key}] {len(df_val)} validated → {out}")


### Step 6 — Validate Initiatives

Runs `run_initiative_validation` on each city's `parent_with_subs` clusters. Saves `{City_Country}_validated_initiatives.csv`.

In [ ]:
# Step 6: Validate Initiatives (LLM call per city)
from utils.initiative_validator import run_initiative_validation

all_df_initiatives = {}  # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    print(f"\n[{key}] Validating initiatives...")

    df_inits = run_initiative_validation(all_clusters[key], verbose=True)
    all_df_initiatives[key] = df_inits

    out = OUTPUT_DIR / f"{key}_validated_initiatives.csv"
    df_inits.to_csv(out, index=False)
    print(f"[{key}] {len(df_inits)} initiatives → {out}")


Total clusters:        25
  Parent+sub clusters: 14
  Individual clusters: 11
  Orphaned subs:       0

[Parent 1] [Pillar 1 | Strategy 1. Retrofit buildings]
  Pillar 1 Strategy 1: Retrofit buildings with specified targets across residentia
  └─ Sub 1: Retrofit residential buildings with 4 or fewer units: 20% by 2030 and 
  └─ Sub 2: Retrofit 20% of total 5+ unit residential buildings by 2030.
  └─ Sub 3: Retrofit 20% of total industrial buildings by 2030.
  └─ Sub 4: Retrofit 90% of total City-owned and sister agency-owned buildings by 
  └─ Sub 5: Retrofit 20% of total commercial buildings by 2035.
[Parent 2] [Pillar 1 | Strategy 2. Connect communities to renewable energy]
  Pillar 1 Strategy 2: Connect communities to renewable energy with targets for co
  └─ Sub 1: Install 5 megawatts of co-owned community solar projects by 2025.
  └─ Sub 2: Increase Chicago-based community renewables to 20 megawatts by 2025.
  └─ Sub 3: Increase community renewables subscriptions to achieve 25% su

### Step 7 — Export Combined Results

Builds the final combined table per city and writes `combined_policies.csv`, `trace_individual_policies.csv`, and `trace_initiative_policies.csv` under `outputs/{City_Country}/`.

In [ ]:
# Step 7: Export Combined Results per City
from utils.exports import build_combined_policies_table, export_combined_table_and_traces

all_combined = {}  # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    city_output_dir = OUTPUT_DIR / key
    df_inits = all_df_initiatives.get(key, pd.DataFrame())

    combined = build_combined_policies_table(
        df_policies=all_df_policies[key],
        df_final_individual=all_df_final[key],
        df_initiatives=df_inits,
        policy_clusters=all_clusters[key],
    )
    all_combined[key] = combined

    paths = export_combined_table_and_traces(
        combined=combined,
        df_initiatives=(df_inits if len(df_inits) else None),
        df_final_individual=all_df_final[key],
        output_dir=city_output_dir,
    )

    print(f"\n[{key}] Exports written to {city_output_dir}/:")
    for k, v in paths.items():
        print(f"  {k}: {v}")


---

## 📊 Summary

Counts across all cities after the full pipeline has run.

In [ ]:
# Summary: Batch Pipeline Results
print("=" * 60)
print("  BATCH PIPELINE SUMMARY")
print("=" * 60)

for entry in BATCH:
    key = city_key(entry["metadata"])
    n_extracted   = len(all_extracted.get(key, []))
    n_clusters    = len(all_clusters.get(key, []))
    n_individual  = len(all_df_final.get(key, pd.DataFrame()))
    n_initiatives = len(all_df_initiatives.get(key, pd.DataFrame()))
    n_combined    = len(all_combined.get(key, pd.DataFrame()))

    print(f"\n  {key}")
    print(f"    Extracted policies:   {n_extracted}")
    print(f"    Clusters:             {n_clusters}")
    print(f"    Individual validated: {n_individual}")
    print(f"    Initiatives:          {n_initiatives}")
    print(f"    Combined rows:        {n_combined}")
    print(f"    Output dir:           outputs/{key}/")

print("\n" + "=" * 60)


Total policy records: 73
Cluster breakdown:
cluster_type      role      
individual        individual    11
parent_with_subs  parent        14
                  sub           48


,cluster_id,cluster_type,role,section_header,sector,policy_statement,parent_statement,verbatim_text,extraction_rationale,policy_type,parent_policy_name
0,0,individual,individual,Letter from the MAYOR,Cross-sector,Invest $188 million in equity-focused climate ...,None,To kickstart the implementation of my administ...,Explicit resource allocation ($188 million) ti...,individual,None
1,1,individual,individual,TREE EQUITY,AFOLU,"Plant 75,000 trees by 2026.",None,Together with city departments and community p...,"Quantified target with deadline (75,000 trees ...",individual,None
2,2,individual,individual,GHG REDUCTION TARGETS / ACCOUNTABILITY AND IMP...,Cross-sector,Reduce Chicago's GHG emissions by a minimum of...,None,The 2022 CAP aims to chart an equitable path t...,Quantifiable economy-wide emissions reduction ...,individual,None
3,3,individual,individual,The Role of Offsets,Cross-sector,Do not count carbon offset credits toward Chic...,None,Chicago will not count offset credits toward i...,Clear policy rule restricting use of offsets f...,individual,None
4,4,individual,individual,CLIMATE FINANCING AND DELIVERY CAPACITY,Buildings,Benchmarking Ordinance (2013) requiring covere...,None,"Benchmarking Ordinance, 2013: This ordinance s...",Binding mechanism (ordinance) with clear requi...,individual,None
5,5,individual,individual,CLIMATE FINANCING AND DELIVERY CAPACITY,Transport,EV Readiness Ordinance (2020) requiring specif...,None,"EV Readiness Ordinance, 2020: This ordinance r...",Binding mechanism (ordinance) with explicit pe...,individual,None
6,6,individual,individual,Chicago Recovery Plan funding overlap with Chi...,Buildings,Chicago Recovery Plan climate investment: Deca...,None,Decarbonize Affordable Multifamily Buildings |...,Resource allocation ($6M) with quantified deli...,individual,None
7,7,individual,individual,Chicago Recovery Plan funding overlap with Chi...,Buildings,Chicago Recovery Plan climate investment: Low-...,None,Low- or Moderate-Income (LMI) Housing Retrofit...,Resource allocation ($15M) with quantified del...,individual,None
8,8,individual,individual,Chicago Recovery Plan funding overlap with Chi...,AFOLU,Chicago Recovery Plan climate investment: Expa...,None,Expand Canopy Coverage | $46 million | Plant 7...,Budgeted program with quantified deliverable (...,individual,None
9,9,parent_with_subs,parent,Pillar 1 | Strategy 1. Retrofit buildings,Buildings,Pillar 1 Strategy 1: Retrofit buildings with s...,None,1 Increase access to utility savings and renew...,A named strategy with multiple lettered sub-ac...,parent,None
